# 🇰🇷 여기선 v5 — 한국 YOLO 학습 (Google Colab)

> AI Hub #71385 생활폐기물 이미지 데이터셋으로 YOLOv8 Small 학습
> 9개 카테고리: 캔/비닐/종이/종이팩/페트/유리/플라스틱/스티로폼/건전지
> 목표: TF.js 변환된 40MB 모델 + 90%+ 정확도

**사용법**: 위에서부터 셀 순서대로 실행 (Shift+Enter)

**필요한 것**:
- AI Hub 계정 (이미 승인됨)
- Colab T4 GPU 활성화 (런타임 → 런타임 유형 변경)

## 0. 환경 셋업

In [ ]:
# GPU 확인
!nvidia-smi

In [ ]:
# 필수 라이브러리
!pip install -q ultralytics==8.2.* tensorflowjs==4.* onnx onnxruntime

## 1. AI Hub Shell 다운로드 + 인증

AI Hub 계정 정보 입력 (이미 #71385 승인됨)

In [ ]:
import os
from getpass import getpass

# AI Hub Shell 다운로드
!wget -q https://api.aihub.or.kr/api/aihubshell.do -O /usr/local/bin/aihubshell
!chmod +x /usr/local/bin/aihubshell

# 인증 정보 입력 (Colab 입력창에 입력, 코드에 저장 X)
AIHUB_ID = input('AI Hub 아이디: ')
AIHUB_PW = getpass('AI Hub 비밀번호: ')

# 환경변수로 설정
os.environ['AIHUB_ID'] = AIHUB_ID
os.environ['AIHUB_PW'] = AIHUB_PW

print('✅ 인증 정보 설정됨')

## 2. 데이터 다운로드 (부분, ~20GB)

어플리케이션 이미지 데이터셋 중 일부만 (전체 570GB X)

In [ ]:
# AI Hub 데이터셋 #71385 파일 목록 조회
!aihubshell -mode l -datasetkey 71385

In [ ]:
# 부분 다운로드 (어플리케이션 이미지 일부)
# 실제 filekey는 위 셀 출력에서 확인하고 아래에 입력
# 예: 어플리케이션 폴더의 첫 번째 part 만
import subprocess

# Colab 디스크 사용량 체크
!df -h /content

# 다운로드 (filekey는 실제 값으로 교체)
# !aihubshell -mode d -datasetkey 71385 -filekey <KEY>

print('⚠️ 위 목록에서 어플리케이션 이미지의 파일 키를 확인 후 -filekey 값에 입력')
print('   디스크 여유 확인 후 부분 다운로드 권장 (~20-30GB)')

## 3. 데이터 전처리 — AI Hub 어노테이션을 YOLO 형식으로 변환

In [ ]:
import json
import os
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split

# AI Hub 클래스 매핑 → YOLO 클래스 ID
CLASSES = {
    'c_1': 0,        # 종이
    'c_2_01': 1,     # 종이팩
    'c_3': 2,        # 캔류
    'c_5_02': 3,     # 페트
    'c_6': 4,        # 플라스틱
    'c_7': 5,        # 비닐
    'c_4_02': 6,     # 유리병
    'c_8_01': 7,     # 흰색 스티로폼
    'c_9': 8,        # 건전지
}
CLASS_NAMES = ['paper', 'paper_pack', 'can', 'pet', 'plastic', 'vinyl', 'glass', 'styrofoam', 'battery']

DATA_ROOT = '/content/aihub_data'  # 다운로드한 데이터 경로
YOLO_ROOT = '/content/yolo_data'   # YOLO 형식 변환 저장

def convert_annotation(json_path, img_w, img_h):
    """AI Hub JSON → YOLO txt 변환"""
    with open(json_path) as f:
        d = json.load(f)
    lines = []
    for obj in d.get('objects', []):
        cls_name = obj.get('class_name', '')
        # 클래스 매핑
        cls_id = None
        for prefix, cid in CLASSES.items():
            if cls_name.startswith(prefix):
                cls_id = cid
                break
        if cls_id is None:
            continue
        ann = obj.get('annotation', [{}])[0]
        coord = ann.get('coord', {})
        x = coord.get('x', 0); y = coord.get('y', 0)
        w = coord.get('width', 0); h = coord.get('height', 0)
        # YOLO 정규화 (center_x, center_y, w, h, 0~1)
        cx = (x + w/2) / img_w
        cy = (y + h/2) / img_h
        nw = w / img_w; nh = h / img_h
        if 0 <= cx <= 1 and 0 <= cy <= 1 and nw > 0 and nh > 0:
            lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
    return lines

print('변환 함수 준비 완료. 실제 데이터 경로에 맞춰 아래 셀 실행.')

In [ ]:
# 데이터 분할 + 변환 (실제 경로 확인 후 실행)
# AI Hub 데이터 구조에 따라 경로 조정 필요

# train/val/test 8:1:1
from glob import glob

# 예시 경로 (실제로는 다운로드된 구조에 맞춤)
img_paths = sorted(glob(f'{DATA_ROOT}/**/*.jpg', recursive=True))
print(f'총 이미지: {len(img_paths)}')

train, temp = train_test_split(img_paths, test_size=0.2, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)
print(f'train: {len(train)}, val: {len(val)}, test: {len(test)}')

# YOLO 폴더 구조 생성
for split in ['train', 'val', 'test']:
    os.makedirs(f'{YOLO_ROOT}/images/{split}', exist_ok=True)
    os.makedirs(f'{YOLO_ROOT}/labels/{split}', exist_ok=True)

# ... (실제 변환 루프는 데이터 구조 확인 후 적용)

## 4. dataset.yaml 작성 + YOLOv8 학습

In [ ]:
# dataset.yaml
dataset_yaml = f'''
path: {YOLO_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: paper
  1: paper_pack
  2: can
  3: pet
  4: plastic
  5: vinyl
  6: glass
  7: styrofoam
  8: battery
'''

with open(f'{YOLO_ROOT}/dataset.yaml', 'w') as f:
    f.write(dataset_yaml)

print(dataset_yaml)

In [ ]:
# YOLOv8 Small 학습 (Colab T4 GPU, 3-5일 예상)
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # pretrained

results = model.train(
    data=f'{YOLO_ROOT}/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='yeoguiseon_v5',
    patience=15,        # early stopping
    save=True,
    plots=True,
    device=0,            # GPU
)

# 학습 결과: /content/runs/detect/yeoguiseon_v5/weights/best.pt

## 5. 정확도 검증 (95% 기준)

In [ ]:
# Test셋 검증
model = YOLO('/content/runs/detect/yeoguiseon_v5/weights/best.pt')
metrics = model.val(data=f'{YOLO_ROOT}/dataset.yaml', split='test')

print(f'mAP@0.5: {metrics.box.map50:.3f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.3f}')
print(f'Precision: {metrics.box.mp:.3f}')
print(f'Recall: {metrics.box.mr:.3f}')

# 클래스별 정확도
for i, name in enumerate(CLASS_NAMES):
    if i < len(metrics.box.maps):
        print(f'  {name}: {metrics.box.maps[i]:.3f}')

## 6. TF.js 변환 (브라우저에서 작동)

In [ ]:
# 1) ONNX로 export
model.export(format='onnx', dynamic=False, simplify=True)

# 2) ONNX → TF SavedModel → TF.js
!pip install -q onnx-tf tensorflow
!onnx-tf convert -i /content/runs/detect/yeoguiseon_v5/weights/best.onnx -o /content/best_tf

# 3) TF.js 변환 (양자화 적용 → 모델 크기 줄임)
!tensorflowjs_converter \
    --input_format=tf_saved_model \
    --output_format=tfjs_graph_model \
    --quantize_uint8 \
    /content/best_tf \
    /content/yeoguiseon_yolo_tfjs

# 4) 크기 확인
!du -h /content/yeoguiseon_yolo_tfjs

## 7. 결과 다운로드 (40MB 모델 파일)

In [ ]:
# zip으로 묶어서 다운로드
!zip -r /content/yeoguiseon_yolo_tfjs.zip /content/yeoguiseon_yolo_tfjs

from google.colab import files
files.download('/content/yeoguiseon_yolo_tfjs.zip')

print('✅ 학습 완료! zip 파일을 받아서 yeoguiseon-v4/models/ 폴더에 풀어 넣으세요.')
print('   GitHub Release로 업로드하면 PWA가 자동으로 로드합니다.')